In [0]:
df_silver_date = spark.read.table("he_prod_incen_analytics_dbw_01.silver.dim_date")
df_gold_month = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_month")

In [0]:
from pyspark.sql.functions import date_format, col, month, year

# 1. Create Surrogate Key: FORMAT DATE to 'YYYYMMDD' as INT
df_dim_date = df_silver_date.withColumn("date_sk", date_format(col("date"), "yyyyMMdd").cast("int"))

# 2. LOOKUP month_sk FROM Dim_Month (Snowflake FK Link)
# We join on the date's year and month matching the month_sk's year and month
df_dim_date = df_dim_date.join(
    df_gold_month,
    (date_format(df_dim_date["date"], "yyyyMM").cast("int") == df_gold_month["month_sk"]),
    "left"
)

# 3. Select ONLY the columns required in Gold (Column Pruning)
df_dim_date = df_dim_date.select(
    "date_sk",
    "date",
    "month_sk",
    "day_of_week",
    "quarter",
    "year",
    "is_holiday"
)

display(df_dim_date.limit(5))

In [0]:
gold_table_fqn = "he_prod_incen_analytics_dbw_01.gold.dim_date"

df_dim_date.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(gold_table_fqn)

print(f"✅ Successfully wrote Dim_Date to Gold layer.")